In [1]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score
)

import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Layer
)

from tensorflow.keras.callbacks import EarlyStopping

In [2]:
X = np.load("../processed/X_no_norm.npy")
y = np.load("../processed/y_no_norm.npy")
subjects = np.load("../processed/subjects_no_norm.npy")

print(X.shape)
print(y.shape)

(4635, 7, 297)
(4635,)


In [3]:
class AttentionLayer(Layer):

    def __init__(self):
        super().__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            name="attention_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):

        score = tf.tanh(
            tf.matmul(inputs, self.W) + self.b
        )

        weights = tf.nn.softmax(
            score,
            axis=1
        )

        context = tf.reduce_sum(
            weights * inputs,
            axis=1
        )

        return context

In [4]:
def build_attention_model():

    inputs = Input(
        shape=(7,297)
    )

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = AttentionLayer()(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [5]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [6]:
# Fold 1 only

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[
    train_sub_idx
]

test_subjects = unique_subjects[
    test_sub_idx
]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:")
print(X_train.shape)

print("\nTest Shape:")
print(X_test.shape)

print("\nTrain Labels:")
print(
    np.unique(
        y_train,
        return_counts=True
    )
)

print("\nTest Labels:")
print(
    np.unique(
        y_test,
        return_counts=True
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape:
(4140, 7, 297)

Test Shape:
(495, 7, 297)

Train Labels:
(array([0, 1]), array([2083, 2057]))

Test Labels:
(array([0, 1]), array([254, 241]))


In [7]:
scaler = StandardScaler()

X_train_flat = X_train.reshape(-1,297)
X_test_flat = X_test.reshape(-1,297)

X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)

X_train = X_train_flat.reshape(X_train.shape)
X_test = X_test_flat.reshape(X_test.shape)

In [8]:
model = build_attention_model()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)


Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - accuracy: 0.6838 - loss: 0.5741 - val_accuracy: 0.7174 - val_loss: 0.5091
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.7657 - loss: 0.4609 - val_accuracy: 0.7754 - val_loss: 0.5184
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.8132 - loss: 0.3798 - val_accuracy: 0.7585 - val_loss: 0.5363
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.8352 - loss: 0.3394 - val_accuracy: 0.7657 - val_loss: 0.5230
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.8731 - loss: 0.2773 - val_accuracy: 0.7536 - val_loss: 0.6369
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.8943 - loss: 0.2141 - val_accuracy: 0.7850 - val_loss: 0.7124


In [9]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

pred = (
    pred > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
              precision    recall  f1-score   support

           0       0.72      0.84      0.78       254
           1       0.80      0.66      0.72       241

    accuracy                           0.75       495
   macro avg       0.76      0.75      0.75       495
weighted avg       0.76      0.75      0.75       495



In [10]:
acc_list = []
prec_list = []
rec_list = []

fold = 1

for train_sub_idx, test_sub_idx in subject_kfold.split(
    unique_subjects
):

    print(f"\n========== Fold {fold} ==========")

    train_subjects = unique_subjects[
        train_sub_idx
    ]

    test_subjects = unique_subjects[
        test_sub_idx
    ]

    train_mask = np.isin(
        subjects,
        train_subjects
    )

    test_mask = np.isin(
        subjects,
        test_subjects
    )

    X_train = X[train_mask]
    y_train = y[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    print("Train:", X_train.shape)
    print("Test :", X_test.shape)

    # --------------------------------
    # Standardization
    # --------------------------------

    scaler = StandardScaler()

    X_train_flat = X_train.reshape(
        -1,
        297
    )

    X_test_flat = X_test.reshape(
        -1,
        297
    )

    X_train_flat = scaler.fit_transform(
        X_train_flat
    )

    X_test_flat = scaler.transform(
        X_test_flat
    )

    X_train = X_train_flat.reshape(
        X_train.shape
    )

    X_test = X_test_flat.reshape(
        X_test.shape
    )

    # --------------------------------
    # Model
    # --------------------------------

    model = build_attention_model()

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    # --------------------------------
    # Prediction
    # --------------------------------

    pred = model.predict(
        X_test,
        verbose=0
    )

    pred = (
        pred > 0.5
    ).astype(int)

    acc = accuracy_score(
        y_test,
        pred
    )

    prec = precision_score(
        y_test,
        pred
    )

    rec = recall_score(
        y_test,
        pred
    )

    print(
        f"Accuracy : {acc:.4f}"
    )

    print(
        f"Precision: {prec:.4f}"
    )

    print(
        f"Recall   : {rec:.4f}"
    )

    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)

    fold += 1


========== Fold 1 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7778
Precision: 0.7610
Recall   : 0.7925

========== Fold 2 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7616
Precision: 0.7661
Recall   : 0.7600

========== Fold 3 ==========
Train: (4140, 7, 297)
Test : (495, 7, 297)
Accuracy : 0.7879
Precision: 0.7874
Recall   : 0.7968

========== Fold 4 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7467
Precision: 0.7004
Recall   : 0.8462

========== Fold 5 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7289
Precision: 0.7143
Recall   : 0.7466

========== Fold 6 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7311
Precision: 0.7056
Recall   : 0.7848

========== Fold 7 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7467
Precision: 0.7806
Recall   : 0.6830

========== Fold 8 ==========
Train: (4185, 7, 297)
Test : (450, 7, 297)
Accuracy : 0.7022
Precision: 0.6802
Re

In [11]:
print("\n========== FINAL ==========")

print(
    "Mean Accuracy:",
    np.mean(acc_list)
)

print(
    "Mean Precision:",
    np.mean(prec_list)
)

print(
    "Mean Recall:",
    np.mean(rec_list)
)

print()

print(
    "Accuracy SD:",
    np.std(acc_list)
)

print(
    "Precision SD:",
    np.std(prec_list)
)

print(
    "Recall SD:",
    np.std(rec_list)
)


========== FINAL ==========
Mean Accuracy: 0.7453939393939394
Mean Precision: 0.7326902110529611
Mean Recall: 0.7708312461084466

Accuracy SD: 0.025404480185763016
Precision SD: 0.0378589636031839
Recall SD: 0.03967286735261864


In [12]:
class AttentionLayerV2(tf.keras.layers.Layer):

    def __init__(self):
        super().__init__()

    def build(self, input_shape):

        hidden_size = input_shape[-1]

        # Ws
        self.W = self.add_weight(
            shape=(hidden_size, hidden_size),
            initializer="glorot_uniform",
            trainable=True,
            name="Ws"
        )

        # bs
        self.b = self.add_weight(
            shape=(hidden_size,),
            initializer="zeros",
            trainable=True,
            name="bs"
        )

        # Final projection
        self.V = self.add_weight(
            shape=(hidden_size, 1),
            initializer="glorot_uniform",
            trainable=True,
            name="V"
        )

    def call(self, inputs):

        # inputs = (batch,7,256)

        u = tf.tanh(

            tf.tensordot(
                inputs,
                self.W,
                axes=1
            ) + self.b

        )

        scores = tf.tensordot(

            u,
            self.V,
            axes=1

        )

        weights = tf.nn.softmax(

            scores,
            axis=1

        )

        context = tf.reduce_sum(

            weights * inputs,
            axis=1

        )

        return context

In [13]:
def build_attention_model_v2():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = AttentionLayerV2()(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [14]:
# ==========================
# Fold 1
# ==========================

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[train_sub_idx]
test_subjects = unique_subjects[test_sub_idx]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

# --------------------------
# Create Train/Test Sets
# --------------------------

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# --------------------------
# Standardization
# --------------------------

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, 297)
X_test_flat = X_test.reshape(-1, 297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

# --------------------------
# Build Model
# --------------------------

model = build_attention_model_v2()

model.summary()

# --------------------------
# Train
# --------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(

    X_train,
    y_train,

    validation_split=0.1,

    epochs=100,

    batch_size=32,

    callbacks=[early_stop]

)

# --------------------------
# Prediction
# --------------------------

pred = model.predict(
    X_test
)

pred = (
    pred > 0.5
).astype(int)

# --------------------------
# Results
# --------------------------

print(
    classification_report(
        y_test,
        pred
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape: (4140, 7, 297)
Test Shape : (495, 7, 297)


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_33 (LSTM)                  │ (None, 7, 256)         │       567,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_33 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_34 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_35 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_35 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_v2              │ (None, 256)            │        66,048 │
│ (AttentionLayerV2)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,684,225 (6.42 MB)

 Trainable params: 1,684,225 (6.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.6871 - loss: 0.5730 - val_accuracy: 0.7657 - val_loss: 0.5017
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.7681 - loss: 0.4664 - val_accuracy: 0.7560 - val_loss: 0.4949
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.8078 - loss: 0.3978 - val_accuracy: 0.7512 - val_loss: 0.5044
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.8366 - loss: 0.3434 - val_accuracy: 0.7415 - val_loss: 0.5226
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 90ms/step - accuracy: 0.8680 - loss: 0.2944 - val_accuracy: 0.7585 - val_loss: 0.5297
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 12s 101ms/step - accuracy: 0.8886 - loss: 0.2520 - val_accuracy: 0.7464 - val_loss: 0.6414
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.9109 - loss: 0.2084 - val_accuracy: 0.7536 - val_loss: 0.6808
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step
              precision    recall  f1-score   

In [15]:
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import Dense

class BahdanauAttention(Layer):

    def __init__(self, units):
        super().__init__()

        self.W1 = Dense(units)

        self.V = Dense(1)

    def call(self, values):

        # values = (batch,7,256)

        score = self.V(

            tf.nn.tanh(

                self.W1(values)

            )

        )

        attention_weights = tf.nn.softmax(

            score,

            axis=1

        )

        context_vector = tf.reduce_sum(

            attention_weights * values,

            axis=1

        )

        return context_vector

In [16]:
def build_attention_model_v3():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = BahdanauAttention(256)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [17]:
# ==========================
# Fold 1
# ==========================

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[train_sub_idx]
test_subjects = unique_subjects[test_sub_idx]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

# --------------------------
# Create Train/Test Sets
# --------------------------

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# --------------------------
# Standardization
# --------------------------

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, 297)
X_test_flat = X_test.reshape(-1, 297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

# --------------------------
# Build Model
# --------------------------

model = build_attention_model_v3()

model.summary()

# --------------------------
# Train
# --------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(

    X_train,
    y_train,

    validation_split=0.1,

    epochs=100,

    batch_size=32,

    callbacks=[early_stop]

)

# --------------------------
# Prediction
# --------------------------

pred = model.predict(
    X_test
)

pred = (
    pred > 0.5
).astype(int)

# --------------------------
# Results
# --------------------------

print(
    classification_report(
        y_test,
        pred
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape: (4140, 7, 297)
Test Shape : (495, 7, 297)


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_36 (LSTM)                  │ (None, 7, 256)         │       567,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_37 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_38 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_38 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention              │ (None, 256)            │        66,049 │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,684,226 (6.42 MB)

 Trainable params: 1,684,226 (6.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - accuracy: 0.6833 - loss: 0.5734 - val_accuracy: 0.7319 - val_loss: 0.5398
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.7732 - loss: 0.4534 - val_accuracy: 0.7536 - val_loss: 0.4870
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 72ms/step - accuracy: 0.8089 - loss: 0.3965 - val_accuracy: 0.7609 - val_loss: 0.5080
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - accuracy: 0.8247 - loss: 0.3453 - val_accuracy: 0.7585 - val_loss: 0.5728
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 70ms/step - accuracy: 0.8588 - loss: 0.2976 - val_accuracy: 0.7198 - val_loss: 0.7065
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 71ms/step - accuracy: 0.8921 - loss: 0.2400 - val_accuracy: 0.7488 - val_loss: 0.6779
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 68ms/step - accuracy: 0.9101 - loss: 0.2037 - val_accuracy: 0.7512 - val_loss: 0.7860
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
              precision    recall  f1-score   su

In [18]:
from tensorflow.keras.layers import Layer
import tensorflow as tf

class FeatureAttention(Layer):

    def __init__(self):
        super().__init__()

    def build(self, input_shape):

        hidden = input_shape[-1]      # 256

        self.W = self.add_weight(
            shape=(hidden, hidden),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            shape=(hidden,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):

        # inputs = (batch,7,256)

        # Feature scores
        scores = tf.tanh(

            tf.tensordot(
                inputs,
                self.W,
                axes=1
            ) + self.b

        )

        # Softmax over FEATURES
        weights = tf.nn.softmax(
            scores,
            axis=-1
        )

        # Weight each feature
        weighted = weights * inputs

        # Aggregate all 7 windows
        context = tf.reduce_sum(
            weighted,
            axis=1
        )

        return context

In [19]:
def build_attention_model_v4():

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = FeatureAttention()(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    model.compile(

        optimizer="adam",

        loss="binary_crossentropy",

        metrics=["accuracy"]

    )

    return model

In [20]:
# ==========================
# Fold 1
# ==========================

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[train_sub_idx]
test_subjects = unique_subjects[test_sub_idx]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

# --------------------------
# Create Train/Test Sets
# --------------------------

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# --------------------------
# Standardization
# --------------------------

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, 297)
X_test_flat = X_test.reshape(-1, 297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

# --------------------------
# Build Model
# --------------------------

model = build_attention_model_v3()

model.summary()

# --------------------------
# Train
# --------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(

    X_train,
    y_train,

    validation_split=0.1,

    epochs=100,

    batch_size=32,

    callbacks=[early_stop]

)

# --------------------------
# Prediction
# --------------------------

pred = model.predict(
    X_test
)

pred = (
    pred > 0.5
).astype(int)

# --------------------------
# Results
# --------------------------

print(
    classification_report(
        y_test,
        pred
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape: (4140, 7, 297)
Test Shape : (495, 7, 297)


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 7, 297)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_39 (LSTM)                  │ (None, 7, 256)         │       567,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_39 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_40 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_40 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_41 (LSTM)                  │ (None, 7, 256)         │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_41 (Dropout)            │ (None, 7, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bahdanau_attention_1            │ (None, 256)            │        66,049 │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,684,226 (6.42 MB)

 Trainable params: 1,684,226 (6.42 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.6884 - loss: 0.5764 - val_accuracy: 0.7222 - val_loss: 0.5283
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.7660 - loss: 0.4696 - val_accuracy: 0.7729 - val_loss: 0.4602
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.8124 - loss: 0.3902 - val_accuracy: 0.7681 - val_loss: 0.4538
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.8376 - loss: 0.3419 - val_accuracy: 0.7560 - val_loss: 0.4913
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.8529 - loss: 0.3060 - val_accuracy: 0.7512 - val_loss: 0.5963
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.8857 - loss: 0.2421 - val_accuracy: 0.7778 - val_loss: 0.5659
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 9s 75ms/step - accuracy: 0.9093 - loss: 0.2063 - val_accuracy: 0.7681 - val_loss: 0.6110
Epoch 8/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.9302 - loss: 0.1667 